# CustomProperty Classes with SequenceOptimizer

This notebook demonstrates how to define your own `CustomProperty` class for use with the `SequenceOptimizer` in GOOSE. You will learn how to:
- Create a custom property class by subclassing `CustomProperty`
- Use it with the optimizer to design sequences with custom constraints
- Analyze the results


In [ ]:
import goose
from goose.optimize import SequenceOptimizer
from goose.backend.optimizer_properties import CustomProperty
import sparrow
from sparrow import Protein


## Define a Custom ProteinProperty

Let's define a property that targets a specific count of Alanine ('A') residues in the sequence. We'll call it `AlanineCount`.

In [ ]:
class AlanineCount(CustomProperty):
    def __init__(self, target_value: float, weight: float = 1.0):
        super().__init__(target_value, weight)
    def calculate_raw_value (self, protein: 'sparrow.Protein') -> float:
        return float(protein.sequence.count('A'))


## Initialize SequenceOptimizer

Create an optimizer instance for a short sequence (e.g., length 30).

In [ ]:
optimizer = SequenceOptimizer(target_length=30, verbose=True)


## Add the Custom Property

Add the `AlanineCount` property to the optimizer, targeting 5 Alanines in the sequence.

In [ ]:
optimizer.add_property(AlanineCount, target_value=5.0, weight=1.0)


## Run the Optimization

Run the optimizer and get the best sequence found.

In [ ]:
custom_sequence = optimizer.run()
print(f'Final sequence alanine count: {custom_sequence.count("A")}')

---

**Tips:**
- You can define any property you want by subclassing `CustomProperty` and implementing the `calculate_raw_value` method.
- You can combine your custom property with built-in properties for multi-objective optimization.
- The `calculate_raw_method` method receives a `sparrow.Protein` object, so you can use any of its methods or attributes.

Try defining other custom properties, such as targeting a specific motif, amino acid fraction, or any sequence-derived metric!

## Combining a custom property with built-in properties

Custom properties can be mixed freely with built-in ones. Here we ask the optimizer to produce a 50-residue, fully disordered, electroneutral sequence that also contains exactly 5 alanines.

In [ ]:
multi_optimizer = SequenceOptimizer(
    target_length=50,
    verbose=True,
    max_iterations=2000,
    seed=0,
)

multi_optimizer.add_property(AlanineCount, target_value=5.0, weight=1.0)
multi_optimizer.add_property(goose.FractionDisorder, target_value=1.0, weight=1.0)
multi_optimizer.add_property(goose.NCPR, target_value=0.0, weight=1.0)

multi_seq = multi_optimizer.run()
prot = Protein(multi_seq)
print(f"Sequence:         {multi_seq}")
print(f"Alanine count:    {multi_seq.count('A')} (target 5)")
print(f"NCPR:             {prot.NCPR:+.3f} (target 0.0)")


## A more sophisticated custom property: aromatic spacing

`calculate_raw_value` receives a `sparrow.Protein`, so any sequence-derived metric can be turned into an optimization target. Below we define `MeanAromaticSpacing` -- the mean residue distance between consecutive aromatic residues (W/F/Y). Larger values mean aromatics are more evenly spaced; smaller values mean they cluster.

We then ask the optimizer for a 60-residue, fully disordered sequence with a target FCR of 0.20 and an aromatic spacing of 8.

In [ ]:
import numpy as np

class MeanAromaticSpacing(CustomProperty):
    """Mean spacing (in residues) between consecutive aromatic residues."""

    def __init__(self, target_value: float, weight: float = 1.0,
                 constraint_type: str = "exact"):
        super().__init__(target_value=target_value, weight=weight,
                         constraint_type=constraint_type)

    def calculate_raw_value(self, protein) -> float:
        seq = protein.sequence
        positions = [i for i, aa in enumerate(seq) if aa in "WFY"]
        if len(positions) < 2:
            # No pairs to space -- return a large value so the optimizer
            # is pushed to introduce aromatics.
            return float(len(seq))
        gaps = np.diff(positions)
        return float(np.mean(gaps))


aromatic_optimizer = SequenceOptimizer(
    target_length=60,
    verbose=True,
    max_iterations=2500,
)

aromatic_optimizer.add_property(MeanAromaticSpacing, target_value=8.0, weight=1.0)
aromatic_optimizer.add_property(goose.FCR, target_value=0.20, weight=1.0)
aromatic_optimizer.add_property(goose.FractionDisorder, target_value=1.0, weight=1.0)

aromatic_seq = aromatic_optimizer.run()
positions = [i for i, aa in enumerate(aromatic_seq) if aa in "WFY"]
print(f"Sequence:        {aromatic_seq}")
print(f"Aromatic positions: {positions}")
if len(positions) >= 2:
    print(f"Mean spacing:    {float(np.mean(np.diff(positions))):.2f} (target 8)")
print(f"FCR:             {Protein(aromatic_seq).FCR:.3f} (target 0.20)")
